# Workshop 1 · Gold KPI Model for Power BI · Solution

![Gold model contract](../../../assets/images/w1_gold_model_contract.png)

This notebook is the reference implementation for Workshop 1. It creates the complete Gold model, validates the grain and prepares the handoff used by the optional Day1_03 checkpoint and Day 2 Power BI module.


## Business Scenario

RetailHub needs one governed Gold contract for revenue, margin, order and product analysis. The model uses conformed dimensions plus a line-grain fact table, then adds a dashboard-friendly wide table for fast report prototyping.


## Target Contract

| Object | Grain | Consumer use |
|---|---|---|
| `gold.dim_date` | one row per calendar date | date relationship and slicers |
| `gold.dim_customer` | one row per customer | customer geography and segment slicers |
| `gold.dim_product` | one row per product | product hierarchy and active flag slicers |
| `gold.fact_sales` | one row per order line | reusable star-schema fact |
| `gold.fact_sales_dashboard` | one row per order line | wide Power BI prototype source |


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running SQL cells.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

This notebook requires the four Silver source tables created by `data/generate_training_dataset.ipynb`.


In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Silver source tables are available.")


### Workshop Validation Helpers

These helper functions keep the task-level assertions short and readable. They are used only for validation, not for the main SQL implementation.


In [ ]:
# -- Validation helpers --
def _table_exists(full_name):
    return spark.catalog.tableExists(full_name)


def _require_table(full_name):
    assert _table_exists(full_name), f"Missing object: {full_name}"


def _columns(full_name):
    return set(spark.table(full_name).columns)


def _require_columns(full_name, required_columns):
    missing = sorted(set(required_columns) - _columns(full_name))
    assert not missing, f"{full_name} missing columns: {missing}"


def _scalar(query):
    row = spark.sql(query).first()
    return row[0] if row is not None else None


def _first(query):
    row = spark.sql(query).first()
    assert row is not None, "Query returned no rows"
    return row


def _print_ok(message):
    print(message)


## 1. Source Baseline

Capture source table sizes before building Gold. Later checks should reconcile to this baseline.


In [ ]:
%sql
SELECT 'customers' AS source_table, COUNT(*) AS rows FROM silver.customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver.products
UNION ALL
SELECT 'sales_orders', COUNT(*) FROM silver.sales_orders
UNION ALL
SELECT 'order_lines', COUNT(*) FROM silver.order_lines
ORDER BY source_table


## 2. Create Gold Schema


Solution note: the schema is idempotent because workshop notebooks and Power BI handoff material expect a stable `gold` contract.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS gold


In [ ]:
# -- Validation --
schema_count = spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE 'gold'").count()
assert schema_count == 1, "Task 1 failed: schema 'gold' was not found in the participant catalog"
_print_ok("Task 1 OK: Gold schema exists and is ready for BI objects.")


Why this works: using one stable `gold` schema gives Power BI developers and downstream notebooks a predictable contract.


## 3. Build `gold.dim_date`

Date dimensions should be generated, not inferred from current facts only. Power BI slicers and incremental refresh filters need stable calendar coverage.


Solution note: the date dimension centralizes calendar labels so Power BI reports do not recreate date logic independently.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_date
COMMENT 'Gold date dimension: one row per calendar day. Power BI relationship target for order_date.'
AS
WITH dates AS (
  SELECT explode(sequence(to_date('2024-01-01'), to_date('2026-12-31'), interval 1 day)) AS date_key
)
SELECT
  date_key,
  year(date_key) AS year,
  quarter(date_key) AS quarter,
  month(date_key) AS month,
  date_format(date_key, 'MMMM') AS month_name,
  date_format(date_key, 'yyyy-MM') AS year_month,
  dayofmonth(date_key) AS day_of_month,
  dayofweek(date_key) AS day_of_week,
  date_format(date_key, 'EEEE') AS day_name,
  CASE WHEN dayofweek(date_key) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM dates


Why this works: the date dimension centralizes calendar logic so Power BI reports do not recreate date labels and slicers independently.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT date_key) AS distinct_dates,
  COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_date_keys,
  MIN(date_key) AS min_date,
  MAX(date_key) AS max_date
FROM gold.dim_date


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_date"
_require_table(table)
_require_columns(table, ["date_key", "year", "quarter", "month", "month_name", "year_month", "day_of_month", "day_of_week", "day_name", "is_weekend"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys, MIN(date_key) AS min_date, MAX(date_key) AS max_date
FROM {GOLD}.dim_date
""")
assert row["rows"] > 0, "Task 2 failed: dim_date is empty"
assert row["duplicate_keys"] == 0, f"Task 2 failed: duplicate date_key rows = {row['duplicate_keys']}"
assert str(row["min_date"]) == "2024-01-01", f"Task 2 failed: min date is {row['min_date']}"
assert str(row["max_date"]) == "2026-12-31", f"Task 2 failed: max date is {row['max_date']}"
_print_ok(f"Task 2 OK: dim_date has {row['rows']} unique calendar dates from 2024-01-01 to 2026-12-31.")


## 4. Build `gold.dim_customer`

Keep a clean customer dimension with business-friendly names. Unknown handling is applied in the dashboard table so source quality issues remain visible.


Solution note: `region AS customer_region` makes the BI field unambiguous and preserves one row per customer for many-to-one relationships.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_customer
COMMENT 'Gold customer dimension: one row per customer, conformed for BI slicers and relationships.'
AS
SELECT
  customer_id,
  customer_name,
  city,
  region AS customer_region,
  country,
  segment,
  created_date
FROM silver.customers


Why this works: the customer dimension exposes one clean relationship key and business-friendly slicer attributes.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_keys
FROM gold.dim_customer


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_customer"
_require_table(table)
_require_columns(table, ["customer_id", "customer_name", "city", "customer_region", "country", "segment", "created_date"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_keys, SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM {GOLD}.dim_customer
""")
assert row["rows"] > 0, "Task 3 failed: dim_customer is empty"
assert row["duplicate_keys"] == 0, f"Task 3 failed: duplicate customer_id rows = {row['duplicate_keys']}"
assert row["null_keys"] == 0, f"Task 3 failed: null customer_id rows = {row['null_keys']}"
_print_ok(f"Task 3 OK: dim_customer has {row['rows']} unique customer keys and BI-ready attributes.")


## 5. Build `gold.dim_product`

Rename price and cost columns so report authors can read the field list without guessing source semantics.


Solution note: product pricing fields are renamed to business terms so report authors can distinguish dimension list price from transactional line price.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.dim_product
COMMENT 'Gold product dimension: one row per product, conformed for BI slicers and category analysis.'
AS
SELECT
  product_id,
  product_name,
  category,
  subcategory,
  unit_cost AS standard_unit_cost,
  unit_price AS list_unit_price,
  is_active
FROM silver.products


Why this works: product attributes are conformed once in Gold, which keeps category and price fields consistent across reports.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT product_id) AS distinct_products,
  COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_keys
FROM gold.dim_product


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_product"
_require_table(table)
_require_columns(table, ["product_id", "product_name", "category", "subcategory", "standard_unit_cost", "list_unit_price", "is_active"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_keys, SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM {GOLD}.dim_product
""")
assert row["rows"] > 0, "Task 4 failed: dim_product is empty"
assert row["duplicate_keys"] == 0, f"Task 4 failed: duplicate product_id rows = {row['duplicate_keys']}"
assert row["null_keys"] == 0, f"Task 4 failed: null product_id rows = {row['null_keys']}"
_print_ok(f"Task 4 OK: dim_product has {row['rows']} unique product keys and conformed attributes.")


## 6. Build `gold.fact_sales`

The fact table is the reusable governed object. Keep natural business keys, conformed relationship keys and additive measures at line grain.


Solution note: the fact stays at line grain and adds status flags so DAX measures can use explicit filters without parsing status text.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales
COMMENT 'Gold sales fact: one row per order line. Shared source for revenue, margin, quantity and order-count KPIs.'
AS
SELECT
  line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  channel,
  status,
  region AS source_region,
  quantity,
  unit_price,
  unit_cost,
  discount_pct,
  line_revenue,
  line_cost,
  line_margin,
  category AS source_category,
  CASE WHEN status = 'Completed' THEN true ELSE false END AS is_completed,
  CASE WHEN status = 'Returned' THEN true ELSE false END AS is_returned,
  CASE WHEN status = 'Cancelled' THEN true ELSE false END AS is_cancelled
FROM silver.order_lines


Why this works: the fact table preserves line grain and keeps reusable revenue, cost and status logic close to the governed data.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM gold.fact_sales


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales"
_require_table(table)
_require_columns(table, ["line_id", "order_id", "order_date", "customer_id", "product_id", "channel", "status", "quantity", "unit_price", "unit_cost", "discount_pct", "line_revenue", "line_cost", "line_margin", "is_completed", "is_returned", "is_cancelled"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids,
       SUM(CASE WHEN line_id IS NULL OR order_id IS NULL OR order_date IS NULL OR customer_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END) AS null_required_keys
FROM {GOLD}.fact_sales
""")
assert row["rows"] > 0, "Task 5 failed: fact_sales is empty"
assert row["duplicate_line_ids"] == 0, f"Task 5 failed: duplicate line_id rows = {row['duplicate_line_ids']}"
assert row["null_required_keys"] == 0, f"Task 5 failed: rows with null required keys = {row['null_required_keys']}"
_print_ok(f"Task 5 OK: fact_sales has {row['rows']} line-grain rows with required keys and measures.")


## 7. Build `gold.fact_sales_dashboard`

This table is intentionally wide and keeps line grain. It is useful for first Power BI reports and stakeholder demos, while the star schema remains the governed model.


Solution note: the wide table is convenient for report prototypes, but left joins and `COALESCE` preserve fact rows and make missing dimension values visible.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_dashboard
COMMENT 'Power BI-ready wide sales table at line_id grain, enriched with date, customer and product attributes.'
AS
SELECT
  f.line_id,
  f.order_id,
  f.order_date,
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  d.year_month,
  d.day_of_week,
  d.day_name,
  d.is_weekend,
  f.customer_id,
  COALESCE(c.customer_name, 'Unknown Customer') AS customer_name,
  COALESCE(c.city, 'Unknown') AS city,
  COALESCE(c.customer_region, 'Unknown') AS customer_region,
  COALESCE(c.country, 'Unknown') AS country,
  COALESCE(c.segment, 'Unknown') AS segment,
  f.product_id,
  COALESCE(p.product_name, 'Unknown Product') AS product_name,
  COALESCE(p.category, f.source_category, 'Unknown') AS category,
  COALESCE(p.subcategory, 'Unknown') AS subcategory,
  COALESCE(p.is_active, false) AS product_is_active,
  f.channel,
  f.status,
  f.quantity,
  f.unit_price,
  f.unit_cost,
  f.discount_pct,
  f.line_revenue,
  f.line_cost,
  f.line_margin,
  f.is_completed,
  f.is_returned,
  f.is_cancelled
FROM gold.fact_sales f
LEFT JOIN gold.dim_date d
  ON f.order_date = d.date_key
LEFT JOIN gold.dim_customer c
  ON f.customer_id = c.customer_id
LEFT JOIN gold.dim_product p
  ON f.product_id = p.product_id


Why this works: the dashboard table is denormalized for simple report pages, but it preserves the same line grain as the fact table.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines
FROM gold.fact_sales_dashboard


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales_dashboard"
_require_table(table)
_require_columns(table, ["line_id", "order_id", "order_date", "year_month", "customer_id", "customer_name", "customer_region", "product_id", "product_name", "category", "subcategory", "channel", "status", "line_revenue", "line_margin", "is_completed", "is_returned", "is_cancelled"])
row = _first(f"""
SELECT (SELECT COUNT(*) FROM {GOLD}.fact_sales) AS fact_rows, COUNT(*) AS dashboard_rows, COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines
FROM {GOLD}.fact_sales_dashboard
""")
assert row["dashboard_rows"] > 0, "Task 6 failed: fact_sales_dashboard is empty"
assert row["duplicate_lines"] == 0, f"Task 6 failed: duplicate dashboard line rows = {row['duplicate_lines']}"
assert row["dashboard_rows"] == row["fact_rows"], f"Task 6 failed: dashboard rows {row['dashboard_rows']} do not match fact rows {row['fact_rows']}"
_print_ok(f"Task 6 OK: fact_sales_dashboard preserves line grain and matches {row['fact_rows']} fact rows.")


## 8. Relationship Readiness

Power BI relationships should be many-to-one from the fact to each dimension. Orphan rows must be zero or explicitly documented as source-quality issues.


In [ ]:
%sql
SELECT 'fact_sales.order_date -> dim_date.date_key' AS relationship, COUNT(*) AS orphan_rows
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales.customer_id -> dim_customer.customer_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales.product_id -> dim_product.product_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id


In [ ]:
# -- Validation --
row = _first(f"""
WITH checks AS (
  SELECT COUNT(*) AS orphan_rows FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_date d ON f.order_date = d.date_key
  UNION ALL SELECT COUNT(*) FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
  UNION ALL SELECT COUNT(*) FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_product p ON f.product_id = p.product_id
)
SELECT SUM(orphan_rows) AS total_orphan_rows FROM checks
""")
assert row["total_orphan_rows"] == 0, f"Task 7 failed: relationship orphan rows = {row['total_orphan_rows']}"
_print_ok("Task 7 OK: fact_sales has zero date/customer/product orphan rows.")


## 9. KPI Smoke Test

These are not final Power BI measures, but they prove that the Gold fields support the expected KPI definitions.


In [ ]:
%sql
SELECT
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct
FROM gold.fact_sales_dashboard


In [ ]:
# -- Validation --
row = _first(f"""
SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
       ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
       COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
       ROUND(100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0), 2) AS margin_rate_pct
FROM {GOLD}.fact_sales_dashboard
""")
assert row["revenue"] is not None and row["revenue"] > 0, "Task 8 failed: completed revenue must be positive"
assert row["gross_margin"] is not None, "Task 8 failed: gross margin is null"
assert row["completed_orders"] > 0, "Task 8 failed: completed order count must be positive"
assert row["margin_rate_pct"] is not None, "Task 8 failed: margin rate is null; protect division with NULLIF"
_print_ok(f"Task 8 OK: KPI smoke test passed for {row['completed_orders']} completed orders.")


## 10. Model Change Impact Check

This section demonstrates the program requirement: a changed filter or consumption shape can change results, performance or both.

The durable Gold contract remains unchanged. The comparison uses `TEMP VIEW` objects for teaching.

Solution note: this comparison separates a business-definition change from a model-shape change. A filter change may alter the KPI; star vs wide should not when the KPI rule is identical.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW w1_status_filter_comparison AS
SELECT
  'completed_only' AS scenario_name,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS order_count
FROM gold.fact_sales_dashboard
UNION ALL
SELECT
  'completed_or_returned',
  ROUND(SUM(CASE WHEN is_completed OR is_returned THEN line_revenue ELSE 0 END), 2),
  COUNT(DISTINCT CASE WHEN is_completed OR is_returned THEN order_id END)
FROM gold.fact_sales_dashboard

Why this works: the temp views separate business-definition changes from model-shape changes without creating durable workshop objects.


In [ ]:
%sql
SELECT *
FROM w1_status_filter_comparison
ORDER BY scenario_name

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW w1_model_pattern_comparison AS
WITH star_schema AS (
  SELECT
    ROUND(SUM(CASE WHEN f.is_completed THEN f.line_revenue ELSE 0 END), 2) AS revenue,
    COUNT(*) AS line_rows
  FROM gold.fact_sales f
  JOIN gold.dim_customer c ON f.customer_id = c.customer_id
  JOIN gold.dim_product p ON f.product_id = p.product_id
),
wide_dashboard AS (
  SELECT
    ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
    COUNT(*) AS line_rows
  FROM gold.fact_sales_dashboard
)
SELECT 'star_schema' AS pattern_name, revenue, line_rows FROM star_schema
UNION ALL
SELECT 'wide_dashboard', revenue, line_rows FROM wide_dashboard

Why this works: both query shapes implement the same KPI rule, so revenue should reconcile even when the model shape differs.


In [ ]:
%sql
SELECT
  'status_filter_delta' AS check_name,
  ROUND(
    MAX(CASE WHEN scenario_name = 'completed_or_returned' THEN revenue END)
    - MAX(CASE WHEN scenario_name = 'completed_only' THEN revenue END),
    2
  ) AS observed_delta
FROM w1_status_filter_comparison
UNION ALL
SELECT
  'star_vs_wide_revenue_delta',
  ROUND(
    MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END)
    - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END),
    2
  )
FROM w1_model_pattern_comparison

In [ ]:
# -- Validation --
status_row = _first("""
SELECT ROUND(MAX(CASE WHEN scenario_name = 'completed_or_returned' THEN revenue END) - MAX(CASE WHEN scenario_name = 'completed_only' THEN revenue END), 2) AS status_filter_delta,
       COUNT(DISTINCT scenario_name) AS scenarios
FROM w1_status_filter_comparison
""")
pattern_row = _first("""
SELECT ROUND(MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END) - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END), 2) AS star_vs_wide_delta,
       COUNT(DISTINCT pattern_name) AS patterns
FROM w1_model_pattern_comparison
""")
assert status_row["scenarios"] == 2, "Task 9 failed: status comparison must contain two scenarios"
assert pattern_row["patterns"] == 2, "Task 9 failed: model pattern comparison must contain two patterns"
assert abs(float(pattern_row["star_vs_wide_delta"])) < 0.01, f"Task 9 failed: star vs wide revenue delta = {pattern_row['star_vs_wide_delta']}"
_print_ok(f"Task 9 OK: status filter delta is {status_row['status_filter_delta']} and star-vs-wide revenue reconciles.")


Expected observation: the status-filter delta is a business-definition change. The star-vs-wide revenue delta should be zero because both query shapes implement the same completed-sales KPI rule.

Trainer note: run the star-schema query and wide-table query separately if you want participants to compare notebook execution times. In Day 2, Query Profile gives the deeper performance view.

## 11. Gold Quality Gate

The SQL checks above are visible for teaching. This assertion cell turns the same expectations into a hard stop for the solution notebook.


In [ ]:
checks = []

queries = {
    "dim_date_duplicate_keys": "SELECT COUNT(*) - COUNT(DISTINCT date_key) AS n FROM gold.dim_date",
    "dim_customer_duplicate_keys": "SELECT COUNT(*) - COUNT(DISTINCT customer_id) AS n FROM gold.dim_customer",
    "dim_product_duplicate_keys": "SELECT COUNT(*) - COUNT(DISTINCT product_id) AS n FROM gold.dim_product",
    "fact_sales_duplicate_line_ids": "SELECT COUNT(*) - COUNT(DISTINCT line_id) AS n FROM gold.fact_sales",
    "dashboard_duplicate_line_ids": "SELECT COUNT(*) - COUNT(DISTINCT line_id) AS n FROM gold.fact_sales_dashboard",
    "date_orphans": "SELECT COUNT(*) AS n FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key",
    "customer_orphans": "SELECT COUNT(*) AS n FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id",
    "product_orphans": "SELECT COUNT(*) AS n FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id",
    "star_wide_revenue_delta_abs": """
        SELECT ABS(ROUND(
          MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END)
          - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END),
          2
        )) AS n
        FROM w1_model_pattern_comparison
    """,
}

for name, query in queries.items():
    observed = spark.sql(query).first()["n"]
    checks.append((name, observed, "OK" if observed == 0 else "FAIL"))

failed = [name for name, observed, _ in checks if observed != 0]
if failed:
    raise Exception(f"Gold quality gate failed: {failed}")

display(spark.createDataFrame(checks, ["check_name", "observed_value", "status"]))


## 12. Object Inventory


In [ ]:
%sql
SELECT 'dim_date' AS object_name, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'fact_sales_dashboard', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name


In [ ]:
# -- Validation --
objects = [f"{GOLD}.dim_date", f"{GOLD}.dim_customer", f"{GOLD}.dim_product", f"{GOLD}.fact_sales", f"{GOLD}.fact_sales_dashboard"]
counts = []
for object_name in objects:
    _require_table(object_name)
    row_count = spark.table(object_name).count()
    assert row_count > 0, f"Task 10 failed: {object_name} has no rows"
    counts.append((object_name, row_count))
_print_ok("Task 10 OK: Gold handoff inventory is complete: " + ", ".join(f"{name}={rows}" for name, rows in counts))


## 13. Power BI Readiness Map

| From table | From column | To table | To column | Cardinality |
|---|---|---|---|---|
| `gold.fact_sales` | `order_date` | `gold.dim_date` | `date_key` | many-to-one |
| `gold.fact_sales` | `customer_id` | `gold.dim_customer` | `customer_id` | many-to-one |
| `gold.fact_sales` | `product_id` | `gold.dim_product` | `product_id` | many-to-one |

For quick demos use `gold.fact_sales_dashboard`. For governed semantic models prefer `gold.fact_sales` plus the dimensions.


Solution note: this map is the relationship contract a Power BI developer should recreate in the semantic model before writing DAX measures.